# 🍜 EDA · Mercado publicitario en el sector gastronómico

> **Dataset primario de auditoría** para análisis de mercado publicitario en el sector gastronómico. Contiene registros crudos (*raw data*) con anomalías simuladas del sector (duplicados, valores nulos y registros vacíos) para evaluar la resiliencia de los flujos de limpieza (ETL) en Python y su posterior ingesta para optimización de presupuestos de pauta y segmentación geoespacial.

| Sesión | Alcance | Estado |
|---|---|---|
| **1** | Estructura del notebook, datasets crudos (CSV + MD), selector de fuente, diagnóstico y limpieza base | ✅ En curso |
| **2** | Puente SQL (`sqlite3`), consultas de negocio, iteraciones de imputación, export para Tableau | ⏳ Pendiente |

📒 Bitácora de negocio y gobernanza: `E:\saoto\Documents\Portafolio\bitacora_restaurantes_publicidad.md`

## 0 · Arquitectura del proyecto (brújula)

```
Portafolio/
├── bitacora_restaurantes_publicidad.md   ← hoja de respuestas externa (negocio + gobernanza)
└── contexto-analisis/
    ├── EDA_Restaurantes_Publicidad.ipynb ← este notebook (guía maestra + ETL)
    ├── restaurantes_raw.csv / .md        ← Hoja 1 · catálogo (con anomalías)
    ├── campana_sprints_raw.csv / .md     ← Hoja 2 · campaña 4 sprints
    └── salidas/                          ← (Sesión 2) datos limpios para Tableau
```

### Hoja 1 · Catálogo (`restaurantes_raw`)

| Columna | Tipo | Descripción |
|---|---|---|
| `id_restaurante` | str | Llave primaria (`R001`…). Une Hoja 1 ↔ Hoja 2 |
| `nombre` | str | Nombre comercial |
| `zona` | str | Zona de influencia (Centro, Norte, Sur, Oriente, Poniente) — *simulada, por validar* |
| `categoria` | str | Tipo de cocina / formato |
| `seguidores_instagram` | int | Seguidores de la cuenta (rango simulado 10k–50k) |
| `engagement_rate` | float | Engagement en **%** (4.1 = 4.1 %) |
| `url_instagram` | str | Llave de empalme con los datos reales de IG |

**Anomalías sembradas a propósito:** espacios al borde (`nombre`, `zona`, `categoria`), 1 duplicado exacto (R002), 1 duplicado que solo aparece tras limpiar espacios (R005), 1 registro totalmente vacío, nulos en `engagement_rate` (R006, R009) y `seguidores_instagram` (R007).

### Hoja 2 · Seguimiento de campaña (`campana_sprints_raw`)
1 mes de pauta (oct-2026) = 4 sprints semanales × 10 restaurantes = 40 filas.

| Grupo | Columnas |
|---|---|
| Identificación | `id_restaurante`, `sprint` (S1–S4), `fecha_inicio`, `fecha_fin` |
| Entradas (se capturan) | `inversion_pauta`, `impresiones`, `alcance`, `clics`, `conversiones`, `ticket_promedio`, `costo_variable_pct` |
| KPIs (se derivan) | `ctr`, `tasa_conversion`, `cpa`, `ingresos_atribuidos`, `margen_contribucion`, `roas`, `roi_publicitario` |

**Fórmulas de negocio**

- `ctr = clics / impresiones`
- `tasa_conversion = conversiones / clics`
- `cpa = inversion_pauta / conversiones`
- `ingresos_atribuidos = conversiones × ticket_promedio`
- `margen_contribucion = ingresos_atribuidos × (1 − costo_variable_pct) − inversion_pauta`
- `roas = ingresos_atribuidos / inversion_pauta`
- `roi_publicitario = margen_contribucion / inversion_pauta`

> Al reemplazar datos simulados por reales basta con capturar las **entradas**; la sección 5 recalcula y audita los KPIs.

## 1 · Configuración y selector de fuente

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

BASE = Path.cwd()

# 🔀 Selector de fuente: "csv" o "md"
FUENTE = "csv"

COLS_NUM_H1 = ["seguidores_instagram", "engagement_rate"]
COLS_TXT_H1 = ["id_restaurante", "nombre", "zona", "categoria", "url_instagram"]

## 2 · Carga de datos (CSV ↔ Markdown)

Ambos lectores leen todo como texto **sin limpiar** (para que las anomalías sobrevivan a la carga) y luego convierten las columnas numéricas con la misma regla.

In [ ]:
def leer_csv(ruta: Path) -> pd.DataFrame:
    # dtype=str conserva los espacios; las celdas vacías llegan como NaN
    return pd.read_csv(ruta, dtype=str, skip_blank_lines=True)


def leer_md(ruta: Path) -> pd.DataFrame:
    """Lee la primera tabla Markdown del archivo.
    Cada celda trae exactamente 1 espacio de relleno a cada lado; lo demás es dato."""
    filas = [l.rstrip("\n") for l in ruta.read_text(encoding="utf-8").splitlines()
             if l.startswith("|")]
    def celdas(linea):
        return [c[1:-1] if len(c) >= 2 else "" for c in linea[1:-1].split("|")]
    encabezado = [c.strip() for c in celdas(filas[0])]
    datos = [celdas(l) for l in filas[2:]]  # filas[1] es el separador |---|
    return pd.DataFrame(datos, columns=encabezado).replace("", np.nan)


def tipar(df: pd.DataFrame, cols_num) -> pd.DataFrame:
    df = df.copy()
    for c in cols_num:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def cargar(nombre_base: str, cols_num, fuente: str = FUENTE) -> pd.DataFrame:
    lectores = {"csv": leer_csv, "md": leer_md}
    if fuente not in lectores:
        raise ValueError(f"FUENTE debe ser 'csv' o 'md', no {fuente!r}")
    return tipar(lectores[fuente](BASE / f"{nombre_base}.{fuente}"), cols_num)


df_raw = cargar("restaurantes_raw", COLS_NUM_H1)
print(f"Fuente: {FUENTE} · {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
df_raw

### 2.1 · Prueba de paridad: ¿Python procesa ambas fuentes de forma indistinta?

In [ ]:
df_csv = cargar("restaurantes_raw", COLS_NUM_H1, "csv")
df_md = cargar("restaurantes_raw", COLS_NUM_H1, "md")
pd.testing.assert_frame_equal(df_csv, df_md)
print("✅ CSV y Markdown producen DataFrames idénticos (valores, tipos y espacios).")

## 3 · Diagnóstico de calidad (antes de limpiar)

In [ ]:
def diagnostico(df: pd.DataFrame) -> pd.Series:
    txt = df.select_dtypes(include="object")
    con_espacios = txt.apply(lambda s: s.str.len().ne(s.str.strip().str.len())).sum().sum()
    return pd.Series({
        "filas": len(df),
        "filas_totalmente_vacias": int(df.isna().all(axis=1).sum()),
        "duplicados_exactos": int(df.duplicated().sum()),
        "celdas_con_espacios_al_borde": int(con_espacios),
        "nulos_seguidores_instagram": int(df["seguidores_instagram"].isna().sum()),
        "nulos_engagement_rate": int(df["engagement_rate"].isna().sum()),
    })

diag_antes = diagnostico(df_raw)
diag_antes

In [ ]:
# Detalle de nulos por columna
df_raw.isna().sum().to_frame("nulos")

## 4 · Limpieza ETL (Hoja 1)

Orden deliberado: **espacios → vacíos → duplicados → nulos**. Si se deduplica antes de recortar espacios, el duplicado de R005 (`"Helados Clemente  "`) pasa desapercibido.

In [ ]:
df = df_raw.copy()

# 4.1 Recortar espacios al borde en columnas de texto
for c in COLS_TXT_H1:
    df[c] = df[c].str.strip()

# 4.2 Eliminar registros totalmente vacíos
df = df.dropna(how="all")

# 4.3 Eliminar duplicados (tras normalizar texto)
df = df.drop_duplicates()
assert df["id_restaurante"].is_unique, "Hay id_restaurante repetidos con datos distintos: revisar a mano"

df.shape

### 4.4 · Gestión de nulos

Decisión de Sesión 1 (provisional, documentar en bitácora): **no perder restaurantes** del catálogo. Se imputa con la mediana de la categoría y, si no existe, la mediana global; una columna bandera deja rastro de qué valor es imputado.

> 🔁 Sesión 2: comparar contra `dropna()` y contra empalme con datos reales antes de fijar la regla.

In [ ]:
for c in COLS_NUM_H1:
    df[f"{c}_imputado"] = df[c].isna()
    mediana_cat = df.groupby("categoria")[c].transform("median")
    df[c] = df[c].fillna(mediana_cat).fillna(df[c].median())

df["seguidores_instagram"] = df["seguidores_instagram"].round().astype("int64")
df = df.sort_values("id_restaurante").reset_index(drop=True)
df_limpio = df
df_limpio

In [ ]:
pd.DataFrame({"antes": diag_antes, "después": diagnostico(df_limpio.drop(columns=[c for c in df_limpio if c.endswith("_imputado")]))})

## 5 · Hoja 2 · Campaña a 4 sprints (carga + auditoría de KPIs)

In [ ]:
COLS_NUM_H2 = ["inversion_pauta", "impresiones", "alcance", "clics", "conversiones",
               "ticket_promedio", "costo_variable_pct", "ctr", "tasa_conversion", "cpa",
               "ingresos_atribuidos", "margen_contribucion", "roas", "roi_publicitario"]

camp = cargar("campana_sprints_raw", COLS_NUM_H2)
camp[["fecha_inicio", "fecha_fin"]] = camp[["fecha_inicio", "fecha_fin"]].apply(pd.to_datetime)
print(camp.shape)
camp.head(8)

In [ ]:
def calcular_kpis(d: pd.DataFrame) -> pd.DataFrame:
    k = pd.DataFrame(index=d.index)
    k["ctr"] = d["clics"] / d["impresiones"]
    k["tasa_conversion"] = d["conversiones"] / d["clics"]
    k["cpa"] = d["inversion_pauta"] / d["conversiones"]
    k["ingresos_atribuidos"] = d["conversiones"] * d["ticket_promedio"]
    k["margen_contribucion"] = k["ingresos_atribuidos"] * (1 - d["costo_variable_pct"]) - d["inversion_pauta"]
    k["roas"] = k["ingresos_atribuidos"] / d["inversion_pauta"]
    k["roi_publicitario"] = k["margen_contribucion"] / d["inversion_pauta"]
    return k

kpis = calcular_kpis(camp)
# Tolerancia relativa del 1 % por redondeos en la captura
desvios = ~np.isclose(camp[kpis.columns], kpis, rtol=0.01)
print("Filas con KPI inconsistente:", int(desvios.any(axis=1).sum()))

In [ ]:
# Integridad referencial Hoja 2 → Hoja 1
huerfanos = set(camp["id_restaurante"]) - set(df_limpio["id_restaurante"])
print("IDs de campaña sin catálogo:", huerfanos or "ninguno ✅")

## 6 · Puente SQL (`sqlite3` en memoria) — ⏳ Sesión 2

Plan:
1. `con = sqlite3.connect(":memory:")`
2. `df_limpio.to_sql("restaurantes", con)` · `camp.to_sql("campana", con)`
3. Primeras consultas de prueba:
   - Inversión, ingresos y ROI por zona.
   - Ranking de restaurantes por `roi_publicitario` acumulado del mes.
   - Evolución S1→S4 del CPA por categoría.

In [ ]:
# ⏳ Sesión 2 — esqueleto
# import sqlite3
# con = sqlite3.connect(":memory:")
# df_limpio.to_sql("restaurantes", con, index=False)
# camp.to_sql("campana", con, index=False)
# pd.read_sql("""
#     SELECT r.zona, SUM(c.inversion_pauta) AS inversion, SUM(c.margen_contribucion) AS margen
#     FROM campana c JOIN restaurantes r USING (id_restaurante)
#     GROUP BY r.zona ORDER BY margen DESC
# """, con)

## 7 · Export para Tableau — ⏳ Sesión 2

- `salidas/restaurantes_limpio.csv` y `salidas/campana_sprints_limpio.csv` (o una tabla unida).
- Pendiente para visualización espacial: agregar `latitud` / `longitud` por restaurante.

In [ ]:
# ⏳ Sesión 2
# SALIDAS = BASE / "salidas"; SALIDAS.mkdir(exist_ok=True)
# df_limpio.to_csv(SALIDAS / "restaurantes_limpio.csv", index=False)
# camp.to_csv(SALIDAS / "campana_sprints_limpio.csv", index=False)

## ✅ Checklist

- [x] Dataset crudo Hoja 1 (CSV + MD) con anomalías
- [x] Dataset Hoja 2 (4 sprints) con KPIs de negocio
- [x] Selector de fuente y prueba de paridad CSV ↔ MD
- [x] Diagnóstico y limpieza base (espacios, vacíos, duplicados, nulos)
- [ ] Empalmar datos reales de Instagram (seguidores, engagement) por `url_instagram`
- [ ] Validar zonas reales y agregar coordenadas
- [ ] Sesión 2: SQL, iteraciones de imputación, export a Tableau